# 06. Prueba de los rangos esperados y de la posible copia de la declaración

Prueba de la señalización de valores anómalos (tarea 7 del backlog, HU-02). Los rangos y los controles están en `src/calidad.py`. Los valores marcados no son eliminados, quedan en la serie para que el Operador pueda revisarlos.

In [1]:
import sys
sys.path.append("../src")

In [2]:
import pandas as pd
from pathlib import Path
from calidad import rango, rangos_temperatura, rangos_cammesa, mw_copia
from carga_cammesa import union_filtro, csv_tabla

## 1. Serie de CAMMESA


In [3]:
rutas = sorted(Path("../data/raw/cammesa").glob("CAMMESA_MEM_ReqMax_*.xlsx"))
tabla = union_filtro(rutas)

In [4]:
tabla.info()

<class 'pandas.DataFrame'>
RangeIndex: 93 entries, 0 to 92
Data columns (total 7 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   MES                        93 non-null     datetime64[us]
 1   MAX_REAL_MW                93 non-null     float64       
 2   POTENCIA_DECLARADA_MW      93 non-null     float64       
 3   EDICION                    93 non-null     str           
 4   DESCARGA                   93 non-null     str           
 5   REVISADO                   93 non-null     bool          
 6   POSIBLE_COPIA_DECLARACION  93 non-null     bool          
dtypes: bool(2), datetime64[us](1), float64(2), str(2)
memory usage: 5.5 KB


## 2. Rangos de temperatura

Los rangos se fijaron con los récords de la estación Trelew Aero del SMN para 1991-2020 (máxima absoluta 42,2 °C, mínima absoluta -12,3 °C, media mensual entre 3,0 y 23,2 °C), con un margen de 3 °C porque ERA5 puede diferir de la estación y después de 2020 ya se registró -13,2 °C. Quedan de -15,3 a 45,2 °C para la máxima y la mínima, y de 0,0 a 26,2 °C para la media.

Para probar que el control funciona se reemplaza un valor por una máxima de 46 °C en una copia de la tabla.

In [5]:
ruta_temp = Path("../data/processed/OPENMETEO_TRELEW_TempMensual.csv")
temperatura = pd.read_csv(ruta_temp, parse_dates=["MES"])
temperatura.info()

<class 'pandas.DataFrame'>
RangeIndex: 93 entries, 0 to 92
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   MES                  93 non-null     datetime64[us]
 1   TEMPERATURA_MEDIA_C  93 non-null     float64       
 2   TEMPERATURA_MAX_C    93 non-null     float64       
 3   TEMPERATURA_MIN_C    93 non-null     float64       
dtypes: datetime64[us](1), float64(3)
memory usage: 3.0 KB


In [6]:
temperatura_anomalos = temperatura.copy()
temperatura_anomalos.loc[30,"TEMPERATURA_MAX_C"] = 46

In [7]:
print("Serie real: ", rango(temperatura,rangos_temperatura).sum())
print("Serie anómala: ", rango(temperatura_anomalos,rangos_temperatura).sum())

Serie real:  0
Serie anómala:  1


## 3. Rangos de demanda

Para la demanda solo se fija un piso, el máximo real y la potencia declarada tienen que ser mayores que 0. No se fija un máximo ya que no hay una fuente que pueda respaldarlo.

In [8]:
print("Meses fuera del rango: ", rango(tabla,rangos_cammesa).sum())

Meses fuera del rango:  0


In [9]:
cammesa_anomalos = tabla.copy()
cammesa_anomalos.loc[30,"MAX_REAL_MW"] = 0
print("Meses fuera del rango: ", rango(cammesa_anomalos,rangos_cammesa).sum())

Meses fuera del rango:  1


## 4. Máximos copiados de la declaración

En la edición 2022-12 se encontraron meses donde el máximo real es casi igual a la potencia declarada, unos 0,003 MW por debajo, la huella de un valor copiado de la declaración en lugar de medido. `mw_copia` marca los meses donde la diferencia es menor que 0,01 MW. La prueba fabrica un caso copiando la declarada en un mes.

In [10]:
cammesa_copia = tabla.copy()
cammesa_copia.loc[30,"MAX_REAL_MW"] = cammesa_copia.loc[30, "POTENCIA_DECLARADA_MW"] - 0.003
print("Serie real: ", mw_copia(tabla).sum())
print("Serie con copia: ", mw_copia(cammesa_copia).sum())

Serie real:  7
Serie con copia:  8


In [11]:
tabla[mw_copia]

,MES,MAX_REAL_MW,POTENCIA_DECLARADA_MW,EDICION,DESCARGA,REVISADO,POSIBLE_COPIA_DECLARACION
16,2020-05-01,41.505,41.510,2023-12,2026-10-24,False,True
17,2020-06-01,42.805,42.809,2023-12,2026-10-24,False,True
18,2020-07-01,41.715,41.720,2023-12,2026-10-24,False,True
19,2020-08-01,40.286,40.291,2023-12,2026-10-24,False,True
20,2020-09-01,39.715,39.719,2023-12,2026-10-24,False,True
21,2020-10-01,38.122,38.125,2023-12,2026-10-24,False,True
22,2020-11-01,39.259,39.263,2023-12,2026-10-24,False,True


In [12]:
tabla["POSIBLE_COPIA_DECLARACION"].sum()

np.int64(7)

In [13]:
ruta_salida = Path("../data/processed/CAMMESA_CTRELEUW_ReqMax.csv")
csv_tabla(tabla, ruta_salida)

## Conclusión

- La serie real no tiene valores fuera de rango, ni en la temperatura ni en la demanda. Los controles detectan el valor anómalo cuando estos existen.
- Siete meses consecutivos, de mayo a noviembre de 2020, tienen la marca de posible copia. La edición 2023-12 no los corrigió respecto de la 2022-12.
- Se conservan en la serie con la marca `POSIBLE_COPIA_DECLARACION`, y se excluyen del entrenamiento de la regresión cuantílica en la etapa de modelado.